<html><div style="font-size:7pt">This notebook may contain text, code and images generated by artificial intelligence. Used model: claude-sonnet-4-20250514, vision model: claude-sonnet-4-20250514, endpoint: None, bia-bob version: 0.34.3.. It is good scientific practice to check the code and results it produces carefully. <a href="https://github.com/haesleinhuepf/bia-bob">Read more about code generation using bia-bob</a></div></html>

# Image Embeddings and UMAP Generation

This notebook demonstrates how to:
1. Generate vision embeddings using the [openai/clip-vit-base-patch32](https://huggingface.co/openai/clip-vit-base-patch32)
2. Create a 3D UMAP visualization of the embeddings
3. Save the results for VEST

In [1]:
import os
import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn.functional as F
from transformers import AutoImageProcessor, AutoModel
from datasets import load_dataset
from umap import UMAP
import random
from pathlib import Path
import requests
from io import BytesIO
import torch

In [2]:
import transformers
transformers.__version__

'4.56.1'

In [3]:
base_dir = Path("./")
images_dir = base_dir / "images"

## Load Vision Embedding Model

In [4]:
from transformers import CLIPProcessor, CLIPModel
import pandas as pd
import torch
from PIL import Image
import requests
import os

# Initialize models
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

## Generate Vision Embeddings for All Images

In [5]:
import os
import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn.functional as F

# Get list of all image files in the images folder and subfolders
image_files = []
for root, dirs, files in os.walk(images_dir):
    for f in files:
        if f.lower().endswith('.png') or f.lower().endswith('.jpg'):
            image_files.append(os.path.join(root, f))

# Initialize lists to store results
filenames = []
embeddings = []
images = []

print(f"{len(image_files)} images found")

396 images found


In [6]:
#image_files = random.sample(image_files, 1000)

In [7]:
print(f"Processing {len(image_files)} images for embeddings...")

# Loop through all image files
for i, image_path in enumerate(image_files):
    # Obtain filename from path
    filename = os.path.relpath(image_path, images_dir)
    
    # Load the image
    current_image = Image.open(image_path)
    images.append(np.asarray(current_image))

    # Apply the processing pipeline
    current_inputs = clip_processor(images=current_image, return_tensors="pt")
    with torch.no_grad():
        current_np_emb = clip_model.get_image_features(**current_inputs).squeeze().tolist()

    # Store results
    filenames.append(filename)
    embeddings.append(current_np_emb)

    if (i + 1) % 100 == 0:
        print(f"Processed {i + 1}/{len(image_files)} images")



# Create DataFrame
df = pd.DataFrame({
    'filename': filenames,
    'embedding': embeddings
})

print(f"Successfully processed {len(df)} images")
display(df.head())

Processing 396 images for embeddings...
Processed 100/396 images
Processed 200/396 images
Processed 300/396 images
Successfully processed 396 images


,filename,embedding
0,001_stadium-district__suburban-greenbelt.png,"[-0.23617148399353027, 0.06485015153884888, 0...."
1,002_roundabout-centered-suburb__stadium-distri...,"[-0.5157508850097656, 0.4254847466945648, 0.24..."
2,003_motorway-corridor__bus-depot-area.png,"[-0.262539267539978, 0.044795624911785126, 0.1..."
3,004_rice-terrace-landscape__peatland.png,"[-0.19610291719436646, -0.09290868043899536, 0..."
4,005_flood-control-levee-corridor__steppe.png,"[-0.18014219403266907, -0.274708092212677, -0...."


## Create 3D UMAP Visualization from Embeddings

In [8]:
# Convert embeddings list to numpy array matrix
embedding_matrix = np.stack(df['embedding'].values)

print(f"Embedding matrix shape: {embedding_matrix.shape}")
print("Creating 3D UMAP coordinates...")

# Create 3D UMAP
umap_reducer = UMAP(n_components=3, random_state=42)
umap_coords_actual = umap_reducer.fit_transform(embedding_matrix)

# Add UMAP coordinates to dataframe
df['x'] = umap_coords_actual[:, 0]
df['y'] = umap_coords_actual[:, 1]
df['z'] = umap_coords_actual[:, 2]

print("UMAP coordinates created successfully")
print(f"X range: {df['x'].min():.2f} to {df['x'].max():.2f}")
print(f"Y range: {df['y'].min():.2f} to {df['y'].max():.2f}")
print(f"Z range: {df['z'].min():.2f} to {df['z'].max():.2f}")
display(df.head())

Embedding matrix shape: (396, 512)
Creating 3D UMAP coordinates...


C:\Users\rober\miniforge3\envs\bob-env\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


UMAP coordinates created successfully
X range: 0.16 to 8.37
Y range: -2.36 to 2.09
Z range: 4.16 to 9.73


,filename,embedding,x,y,z
0,001_stadium-district__suburban-greenbelt.png,"[-0.23617148399353027, 0.06485015153884888, 0....",7.481415,1.142234,7.098687
1,002_roundabout-centered-suburb__stadium-distri...,"[-0.5157508850097656, 0.4254847466945648, 0.24...",7.318924,1.370635,6.661978
2,003_motorway-corridor__bus-depot-area.png,"[-0.262539267539978, 0.044795624911785126, 0.1...",6.691281,1.082104,7.644025
3,004_rice-terrace-landscape__peatland.png,"[-0.19610291719436646, -0.09290868043899536, 0...",3.233606,0.800508,5.303019
4,005_flood-control-levee-corridor__steppe.png,"[-0.18014219403266907, -0.274708092212677, -0....",3.688275,1.494951,5.545282


## Save Results to CSV File

In [9]:
# Save the dataframe to CSV
output_path = base_dir / "data.csv"

# Convert embedding arrays to strings for CSV storage
df_to_save = df.copy()
df_to_save['embedding'] = df_to_save['embedding'].apply(lambda x: ','.join(map(str, x)))

df_to_save.to_csv(output_path, index=False)

print(f"Results saved to {output_path}")
print(f"Final dataframe shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
display(df[['filename', 'x', 'y', 'z']].head(10))

Results saved to data.csv
Final dataframe shape: (396, 5)
Columns: ['filename', 'embedding', 'x', 'y', 'z']


,filename,x,y,z
0,001_stadium-district__suburban-greenbelt.png,7.481415,1.142234,7.098687
1,002_roundabout-centered-suburb__stadium-distri...,7.318924,1.370635,6.661978
2,003_motorway-corridor__bus-depot-area.png,6.691281,1.082104,7.644025
3,004_rice-terrace-landscape__peatland.png,3.233606,0.800508,5.303019
4,005_flood-control-levee-corridor__steppe.png,3.688275,1.494951,5.545282
5,006_volcanic-highlands__power-plant-zone.png,4.324611,1.861881,6.105531
6,007_bog__general-aviation-airfield.png,6.498019,0.731411,8.190401
7,008_village-center__glacier-foreland.png,4.334042,1.490661,6.162693
8,009_bridge-crossing-zone__marina-district.png,8.197562,1.254462,6.688144
9,010_harbor-waterfront__mountain-ridge.png,4.254158,1.624420,6.131436
